<a href="https://colab.research.google.com/github/bsheese/387/blob/main/19_onramp/19_1_ML_Foundations/19_1_4_Softmax_and_XOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The On-Ramp: Part 4
## Softmax, and What One Neuron Can't Do

---

## What This Notebook Is About

In 19_1_3 we saw that the Pets model's last layer computes four scores, one weighted sum per breed. This notebook finishes the path from scores to predictions, and then looks at the limits of what a single layer of neurons can do.

1. **Softmax**: how four scores, which can be any numbers, become four probabilities that add up to 1
2. **The loss**: how training measures how wrong a probability is, using the logarithm
3. **XOR**: a tiny problem with four points that no single neuron can solve
4. **A hidden layer**: two neurons feeding a third solve XOR, which is the reason networks have more than one layer

**Optional reference.** If you took CS 377, you have seen these ideas before in [18_5_1](https://github.com/bsheese/cs377/blob/main/18_classification/18_5_MutliClassClassification/18_5_1_Multiclass_Intro.ipynb), [18_2_3](https://github.com/bsheese/cs377/blob/main/18_classification/18_2_LogisticRegression/18_2_3_LogReg_to_NeuralNetworks.ipynb). If you haven't, those notebooks cover the same ideas on other datasets, in more depth. You don't need them for this course.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

preds = pd.read_csv(fetch('pets4_predictions.csv'))
BREEDS = ['Birman', 'Ragdoll', 'basset_hound', 'beagle']
scores = preds[[f'score_{b}' for b in BREEDS]].to_numpy()
probs = preds[[f'prob_{b}' for b in BREEDS]].to_numpy()
print('scores:', scores.shape, '  probs:', probs.shape)

---

## Section 1: From Scores to Probabilities: Softmax

Here are the four scores and four probabilities that 19_0_2 saved for the first photo, a Birman:

In [ ]:
print('photo:', preds.loc[0, 'file'])
print('scores:       ', scores[0].round(3))
print('probabilities:', probs[0].round(3))

The scores can't be used as probabilities directly. Some are negative, and they don't add up to 1. The function that turns them into probabilities is called **softmax**, and it has two steps:

1. **Make every score positive** by raising $e$ (2.718...) to the power of the score. `np.exp` does this. The bigger the score, the bigger the result, so the order of the breeds doesn't change.
2. **Divide each result by the total**, so the four numbers add up to 1.

Here is how $e^{x}$ behaves for a few values of $x$. It's always positive, it's exactly 1 at $x = 0$, and it grows very quickly as $x$ increases:

In [ ]:
for value in [-4, -2, -1, 0, 1, 2, 4]:
    print(f'e^{value:<2} = {np.exp(value):8.3f}')

Now the two steps on the first photo's scores:

In [ ]:
s = scores[0]
step1 = np.exp(s)
print('step 1, e^score:     ', step1.round(3))
print('total:               ', step1.sum().round(3))
step2 = step1 / step1.sum()
print('step 2, divided:     ', step2.round(3))
print('saved probabilities: ', probs[0].round(3))

They match, so softmax is exactly what the model does to its scores.

To apply softmax to all 800 photos at once, each row has to be divided by its own total. `.sum(axis=1)` adds across each row, and `keepdims=True` keeps the result as a column of shape (800, 1), so that dividing the (800, 4) array by it divides every row by its own total. (This is the broadcasting from 19_1_3.)

In [ ]:
def softmax(scores):
    exps = np.exp(scores)
    return exps / exps.sum(axis=1, keepdims=True)

print('row totals:', np.exp(scores).sum(axis=1, keepdims=True).shape)
mine = softmax(scores)
print('largest difference from the saved probabilities:', np.abs(mine - probs).max().round(6))
print('every row adds up to 1:', np.allclose(mine.sum(axis=1), 1))

### What softmax pays attention to

Softmax only depends on the **differences** between the scores. Adding the same number to all four scores multiplies every $e^{\text{score}}$ by the same amount, which cancels when we divide by the total. But making the differences bigger makes the model more confident:

In [ ]:
s = np.array([[2.0, 1.0, 0.0, -1.0]])
print('scores             ', s[0], ' -> ', softmax(s)[0].round(3))
print('scores + 10        ', (s + 10)[0], ' -> ', softmax(s + 10)[0].round(3))
print('scores x 3         ', (s * 3)[0], ' -> ', softmax(s * 3)[0].round(3))
print('scores x 0.1       ', (s * 0.1)[0], ' -> ', softmax(s * 0.1)[0].round(3))

Multiplying the scores by 3 spreads them apart, and the top breed's probability goes from 0.64 to 0.95. Multiplying by 0.1 squeezes them together, and the probabilities move toward 0.25 each. Unit 23 uses this to control how adventurous a text generator is.

### Sigmoid is softmax with two classes

In 19_1_3, logistic regression used sigmoid to turn a single weighted sum $z$ into a probability. That's the same as softmax applied to two scores, $z$ for "yes" and 0 for "no":

In [ ]:
for z in [-2.0, 0.0, 1.5]:
    two_scores = np.array([[z, 0.0]])
    print(f'z = {z:4}:  softmax gives {softmax(two_scores)[0, 0]:.4f},  sigmoid gives {1 / (1 + np.exp(-z)):.4f}')

So a yes/no model and a four-breed model end the same way: weighted sums, then softmax. With two classes it's usually written as one score and sigmoid.

---

## Section 2: How Wrong Is a Probability? The Loss

19_1_1 plotted the **loss** during training and said only that lower is better. Now we can say what it is. For each photo, take the probability the model gave to the **true** breed. If that probability is close to 1, the model did well; if it's close to 0, the model was confidently wrong. The loss turns this into a penalty with the **natural logarithm**, `np.log`:

$$\text{loss for one photo} = -\log(\text{probability of the true breed})$$

The logarithm undoes $e^{x}$: $\log(e^{x}) = x$. So $\log(1) = 0$, and the log of a number between 0 and 1 is negative, more and more so as the number gets closer to 0. The minus sign makes the penalty positive.

In [ ]:
print('probability of     penalty')
print('the true breed     -log(p)')
for p in [0.99, 0.9, 0.5, 0.25, 0.1, 0.01]:
    print(f'{p:12}       {-np.log(p):6.2f}')

A right answer given with high confidence costs almost nothing, a 50/50 answer costs 0.69, and a confident wrong answer (0.01 for the true breed) costs 4.6. Being confidently wrong is penalized far more than being unsure. This penalty is called the **cross-entropy loss**, and the loss of a whole set of photos is the average of the photos' penalties. It's what `nn.CrossEntropyLoss` computed in 19_0_1 and 19_1_1.

Here it is for the test photos. `probs[rows, cols]` picks out one element per row: for each test photo, the probability in the column of its true breed.

In [ ]:
is_test = preds['split'].to_numpy() == 'test'
true_col = np.array([BREEDS.index(b) for b in preds['breed']])

p_true = probs[is_test, :][np.arange(is_test.sum()), true_col[is_test]]
print('probability of the true breed, first five test photos:', p_true[:5].round(3))
print(f'average loss on the test photos: {(-np.log(p_true)).mean():.3f}')
print(f'worst single photo: p = {p_true.min():.3f}, penalty {-np.log(p_true.min()):.2f}')

Training a network means changing its weights to make this average smaller. Unit 20 shows how.

---

## Section 3: A Problem One Neuron Can't Solve

Now we leave the pets for a tiny made-up problem with two features and four points. The answer is 1 when exactly one of the two features is 1, and 0 otherwise. This pattern is called **XOR** ("exclusive or"). Unit 20 uses these same four points.

In [ ]:
X_xor = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]])
y_xor = np.array([0, 1, 1, 0])

for x, y in zip(X_xor, y_xor):
    print(f'x1 = {x[0]}, x2 = {x[1]}  ->  {y}')

In [ ]:
def plot_xor(ax, title):
    for x, y in zip(X_xor, y_xor):
        ax.scatter(x[0], x[1], s=250, c='#C0392B' if y == 1 else '#2980B9', edgecolors='black', zorder=3)
        ax.annotate(str(y), (x[0], x[1]), xytext=(10, 10), textcoords='offset points', fontsize=13)
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_xor(ax, 'XOR: red = 1, blue = 0')
plt.show()

The two 1s sit on opposite corners of the square, and so do the two 0s.

A single neuron computes $z = w_1 x_1 + w_2 x_2 + b$ and answers 1 when the probability is above 0.5, which is when $z > 0$ (sigmoid of 0 is exactly 0.5). The points where $z = 0$ form a **straight line**. So whatever weights a single neuron has, it splits the plane with one straight line and answers 1 on one side and 0 on the other. The line is called its **decision boundary**.

The next cell draws the boundary for three choices of weights and counts how many of the four points each one gets right.

In [ ]:
def neuron_answers(w1, w2, b):
    z = w1 * X_xor[:, 0] + w2 * X_xor[:, 1] + b
    return (z > 0).astype(int)

attempts = [(1, 1, -0.5), (-1, -1, 1.5), (1, -1, -0.5)]
xs = np.linspace(-0.5, 1.5, 10)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (w1, w2, b) in zip(axes, attempts):
    right = (neuron_answers(w1, w2, b) == y_xor).sum()
    plot_xor(ax, f'w1={w1}, w2={w2}, b={b}:  {right} of 4 right')
    ax.plot(xs, -(w1 * xs + b) / w2, 'k--')    # the line where z = 0
plt.tight_layout()
plt.show()

Each line gets 3 of the 4 points right. To get all four, a line would have to put the two red corners on one side and the two blue corners on the other, and no straight line can do that, because the red corners are diagonally opposite each other. Trying thousands of weights confirms it:

In [ ]:
best = 0
grid = np.linspace(-3, 3, 25)
for w1 in grid:
    for w2 in grid:
        for b in grid:
            best = max(best, (neuron_answers(w1, w2, b) == y_xor).sum())
print(f'tried {len(grid) ** 3:,} combinations of weights; the best got {best} of 4 right')

Logistic regression, the trained single neuron from 19_1_3, does even worse:

In [ ]:
lr = LogisticRegression().fit(X_xor, y_xor)
print('weights:', lr.coef_[0].round(3), '  bias:', lr.intercept_.round(3))
print('probabilities of 1:', lr.predict_proba(X_xor)[:, 1].round(3))
print('accuracy:', lr.score(X_xor, y_xor))

It sets both weights to 0 and gives every point a probability of 0.5. Every line gets at least one point wrong, and because the problem is symmetric there's no reason to prefer one line over another, so the best it can do by the loss from Section 2 is to say "50/50" everywhere. Its accuracy of 0.5 is no better than guessing.

This is not just a puzzle. In 1969 Marvin Minsky and Seymour Papert pointed out that a single-layer network can't learn XOR, and their book helped turn many researchers away from neural networks for over a decade.

---

## Section 4: Adding a Hidden Layer

The way around the problem is to put neurons in front of the neuron that gives the answer. We can build such a network by hand. To keep the arithmetic easy, each neuron here uses a **step** activation instead of sigmoid: it outputs 1 if its weighted sum is above 0 and 0 otherwise.

- **Hidden neuron 1** has weights (1, 1) and bias −0.5. It outputs 1 when **at least one** feature is 1.
- **Hidden neuron 2** has weights (1, 1) and bias −1.5. It outputs 1 only when **both** features are 1.
- **The output neuron** takes the two hidden outputs as its inputs, with weights (1, −1) and bias −0.5. It outputs 1 when hidden neuron 1 is on and hidden neuron 2 is off, which is "at least one, but not both".

In [ ]:
def step(z):
    return (z > 0).astype(int)

h1 = step(X_xor @ np.array([1, 1]) - 0.5)    # at least one is 1
h2 = step(X_xor @ np.array([1, 1]) - 1.5)    # both are 1
H = np.column_stack([h1, h2])                # the hidden layer's outputs, one row per point
out = step(H @ np.array([1, -1]) - 0.5)      # at least one, but not both

print('x1 x2 | h1 h2 | output  target')
for x, h, o, y in zip(X_xor, H, out, y_xor):
    print(f' {x[0]}  {x[1]} |  {h[0]}  {h[1]} |   {o}       {y}')

All four points are right. The two **hidden neurons** form a **hidden layer**. It's called hidden because its outputs are neither the inputs nor the final answer; they are new features that the network computes for itself.

Plotting the points by their hidden-layer values, instead of their original features, shows why this works:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
plot_xor(ax1, 'Original features: no line works')

for h, y in zip(H, y_xor):
    ax2.scatter(h[0], h[1], s=250, c='#C0392B' if y == 1 else '#2980B9', edgecolors='black', zorder=3)
ax2.annotate('(0, 1) and (1, 0)\nboth land here', (1, 0), xytext=(0.3, 0.35), fontsize=10,
             arrowprops=dict(arrowstyle='->'))
xs = np.linspace(-0.5, 1.5, 10)
ax2.plot(xs, xs - 0.5, 'k--')     # the output neuron's line: h1 - h2 - 0.5 = 0
ax2.set_xlim(-0.5, 1.5)
ax2.set_ylim(-0.5, 1.5)
ax2.set_xlabel('h1 (at least one)')
ax2.set_ylabel('h2 (both)')
ax2.set_title("Hidden-layer features: one line works")
plt.tight_layout()
plt.show()

In the hidden layer's features, the two red points land on the same spot, and a single straight line separates them from the two blue points. The hidden layer doesn't solve the problem by itself. It moves the points into a new arrangement where the output neuron's straight line is enough.

That is the main idea behind deep networks, and it applies directly to our Pets model. Its last layer is four neurons, and each one can only draw straight boundaries through the 512 features. That works because the 17 layers in front of it turn photos, where breeds are hopelessly tangled together, into 512 features where breeds can be separated with straight boundaries. Those earlier layers are hidden layers too, a lot of them.

We set the hidden weights here by hand, because we already knew the answer. In unit 20 a network learns the weights of its hidden layer from the four XOR points on its own, and in 22_1 we look at what ResNet's hidden layers have learned.

---

## Summary

- **Softmax** turns a list of scores into probabilities: raise $e$ to the power of each score (making them positive), then divide by the total (making them add up to 1). It reproduces the Pets model's saved probabilities exactly. Only the differences between scores matter, and bigger differences give more confident probabilities.
- **Sigmoid** is softmax with two classes, where one score is fixed at 0.
- The **cross-entropy loss** of a photo is $-\log$ of the probability given to its true breed. It's near 0 for confident right answers and very large for confident wrong ones. Training makes the average loss smaller.
- A single neuron's **decision boundary** is a straight line (with more features, a flat surface). **XOR** can't be separated by any straight line, so no single neuron solves it; logistic regression gives up and answers 0.5 everywhere.
- A **hidden layer** of neurons computes new features. With two hidden neurons ("at least one" and "both"), the XOR points can be separated by a straight line, and the network gets all four right.
- The Pets model works the same way: its hidden layers turn photos into 512 features in which the breeds can be separated by the last layer's straight boundaries.

**Next:** unit 20 opens up how a network learns its weights: tensors, the nudge test, autograd and gradient descent, and then a network that learns XOR by itself.